# Gemma Clean-K2 INT4 Transfer Test & Matched Random Control

**Closing Disclosed Gap (A)**:
This notebook applies Gemma-2-9B's locked parametric clean-K2 pair (`L31_H7, L39_H6`), identified on FP16, unchanged to the canonical GPTQ INT4 checkpoint (`gemma2_9b_base_gptq_int4_ncalib128`, $N_{\mathrm{CALIB}}=128$).

It also runs a matched random-head control ($K=2$, seed 42) for matched comparison:
- **Clean-K2 locked pair**: `L31_H7` and `L39_H6` (FP16 reference point: $-6\ [-12, -1]$).
- **Random heads**: `L41_H11` and `L7_H3` (drawn with seed 42 from all 672 heads [42 layers $\times$ 16 heads], excluding clean K2, context K10, and raw peak L27_H6).
- **Steering protocol**: Dual-run JuICE steering, $\alpha_+=1.0, \alpha_-=0.0$ (parametric enlarge direction), greedy decoding, `max_new_tokens=16`, `prompt_format="juice_coherent"` on the preregistered seed-42 $n=100$ conflict items.
- **Precisions**: Sequentially runs (a) FP16 base (`google/gemma-2-9b`) and (b) canonical INT4 (`gemma2_9b_base_gptq_int4_ncalib128`), clearing VRAM between them.
- **Scoring**: Question-aware matcher + paired bootstrap (1000 resamples, seed 42) reporting both **gold-PAR-only $\Delta$** and **inclusive $\Delta$** extending Table 12.

### Recommended Kaggle Settings
- **Accelerator**: GPU T4 x2 (Free tier)
- **Internet**: On (for Hugging Face model access)
- **Secrets**: Add `HF_TOKEN` with access to `google/gemma-2-9b` (accept Gemma license on Hugging Face)
- **Datasets attached**: 
  1. `kaggle_gemma_clean_k2_int4_transfer.zip` (this package)
  2. `gemma2_9b_base_gptq_int4_ncalib128` (canonical GPTQ INT4 model directory from quantize notebook output)


In [ ]:
# Setup environment, unpack package if attached as zip, authenticate Hugging Face, and inspect GPUs
import os
import sys
import zipfile
from pathlib import Path
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device count: {torch.cuda.device_count()}")
    for d in range(torch.cuda.device_count()):
        print(f"  Device {d}: {torch.cuda.get_device_name(d)} ({(torch.cuda.get_device_properties(d).total_memory / 1e9):.1f} GB)")

# Unpack dataset if zip is present in /kaggle/input
for p in Path("/kaggle/input").rglob("*.zip") if Path("/kaggle/input").exists() else []:
    if "gemma" in str(p).lower() and ("clean_k2" in str(p).lower() or "transfer" in str(p).lower()):
        print(f"Extracting {p} into /kaggle/working...")
        try:
            with zipfile.ZipFile(p, 'r') as zf:
                zf.extractall("/kaggle/working")
        except Exception as e:
            print(f"Extract notice: {e}")

# Hugging Face Authentication via Kaggle Secrets (Add-ons -> Secrets -> HF_TOKEN)
# Required because google/gemma-2-9b is a gated repository
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    hf_token = None
    for k in ['HF_TOKEN', 'hf_token', 'HUGGINGFACE_TOKEN', 'HUGGING_FACE_HUB_TOKEN']:
        try:
            hf_token = sec.get_secret(k)
            if hf_token:
                print(f"[AUTH] Found secret key '{k}' in Kaggle Secrets.")
                break
        except Exception:
            pass
    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        print('[AUTH] Successfully authenticated Hugging Face with Kaggle Secrets.')
    else:
        print('[AUTH] Warning: No HF_TOKEN secret found in Kaggle Secrets.')
except Exception as e:
    print(f'[AUTH] Kaggle Secrets login notice: {e}')

# Install gptqmodel for Kaggle Python 3.12 (required for INT4 GPTQ load)
try:
    import gptqmodel
except ImportError:
    import subprocess
    print('Installing gptqmodel==2.2.0...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gptqmodel==2.2.0', 'transformers==4.51.3', 'accelerate>=0.26.0,<2.0'])
    print('gptqmodel installed successfully.')

working = Path("/kaggle/working")
if str(working) not in sys.path:
    sys.path.insert(0, str(working))

# Find run script
candidates = list(working.rglob("run_gemma_clean_k2_int4_transfer.py"))
if Path("/kaggle/input").exists():
    candidates += list(Path("/kaggle/input").rglob("run_gemma_clean_k2_int4_transfer.py"))
if not candidates:
    candidates = list(Path(".").rglob("run_gemma_clean_k2_int4_transfer.py"))

assert candidates, "Could not find run_gemma_clean_k2_int4_transfer.py"
SCRIPT_PATH = candidates[0]
print(f"Run script resolved: {SCRIPT_PATH}")


In [ ]:
# Run Gemma Clean-K2 INT4 Transfer Test + Matched Random-K2 Control
import os
import subprocess

cmd = [
    sys.executable, "-u", str(SCRIPT_PATH),
    "--fp16-model", "google/gemma-2-9b",
    "--seed", "42",
    "--n-boot", "1000",
]

print("Starting experiment run...")
env = os.environ.copy()
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in iter(proc.stdout.readline, ''):
    print(line, end='', flush=True)
proc.stdout.close()
ret = proc.wait()
assert ret == 0, f"Experiment failed with returncode {ret}"


In [ ]:
# Display Table 12 Extension Results and inspect generated output files
import pandas as pd
from IPython.display import display, Markdown

md_files = list(Path("/kaggle/working").rglob("gemma_clean_k2_int4_transfer_n100_ncalib128.md"))
csv_files = list(Path("/kaggle/working").rglob("gemma_clean_k2_int4_transfer_n100_ncalib128.csv"))

if md_files:
    content = md_files[0].read_text(encoding="utf-8")
    display(Markdown("### Table 12 Extension: Gemma Clean-K2 INT4 Transfer & Matched Controls"))
    display(Markdown(content))

if csv_files:
    df = pd.read_csv(csv_files[0])
    display(df)

print("Done! You can download gemma_clean_k2_int4_transfer_n100_ncalib128.json, .md, and .csv from /kaggle/working.")
